In [7]:
import xarray as xr
import numpy as np

In [8]:
# static features
stat_dict = {}
stat_features = ['clelev', 'clfaotxt', 'cltveg', 'phh2o', 'topsoil_bulk_density', 'vegetation_type_11', 'wetlandtype', 'climatetype']
stat_files = ['clelev.nc', 'clfaotxt.nc', 'cltveg.nc', 'phh2o.nc', 'topsoil_bulk_density.nc', 'vegetation_type_11.nc', 'wetlandtype.nc', 'climatetype.nc']
for i, feat in enumerate(stat_features):
    filename = f'../../data/TEM-MDM/{stat_files[i]}' 
    feature = xr.open_dataset(filename)
    feature = feature[feat].values
    stat_dict[feat] = feature.reshape(-1)

# for index, i in enumerate(stat_dict['clelev']):
#     if not np.isnan(i):
#         print(i)
#         print(index)
#         break

In [9]:
# non-spatial features
nosp_features = ['ch4','co2']
nosp_files = ['ch4-1979-2018','kco21979-2018']
ch4_dict, co2_dict = {}, {}
with open('../../data/TEM-MDM/ch4-1979-2018.txt', 'r') as f:
    for line in f:
        year, ch4 = line.split()
        ch4_dict[int(year)] = float(ch4)
with open('../../data/TEM-MDM/kco21979-2018.txt', 'r') as f:
    for line in f:
        year, co2 = line.split()
        co2_dict[int(year)] = float(co2)

In [10]:
def convert2year(data):
    days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    mask = np.zeros(data.shape, dtype=bool)  
    for month, days in enumerate(days_in_month):
        mask[:, month, :days] = True
    valid_data = data[mask]
    valid_data = valid_data.reshape(data.shape[0], 365)
    return valid_data

In [11]:
mask = np.full((360*720), False, dtype=bool)
for year in range(1979, 2019):
    output_file = f'../../data/TEM-MDM/CH4_emission_intensity_{year}.nc'
    label = xr.open_dataset(output_file, engine="netcdf4")
    y_data = convert2year(label['CH4_emission'].values.reshape(-1,12,31))
    mask = mask | np.isnan(y_data).any(axis=1)

In [12]:
indices = np.random.permutation((~mask).sum())  
fold_index = np.array_split(indices, 5)

In [13]:
input_vars = ['daily_ecmwf_PREC', 'daily_ecmwf_SOLR', 'daily_ecmwf_TAIR', 'daily_ecmwf_VAPR', 'monthly_NPP'] 
input_vars_names = ['PREC', 'SOLR', 'TAIR', 'VAPR', 'NPP'] 
data_dict = {}
shape = (360*720, 365)
shape_masked = (360*720, 365)
all_mask_true_ids = []
for year in range(1979, 2019):
    print(year)
    X_data = []
    for feat in stat_dict.keys():
        feature = stat_dict[feat]
        feature = np.repeat(feature[:, np.newaxis], 365, axis=1)
        X_data.append(feature)
    feature = np.full(shape, ch4_dict[year])
    X_data.append(feature)
    feature = np.full(shape, co2_dict[year])
    X_data.append(feature)
    for i,input_var in enumerate(input_vars):
        input_file = f'../../data/TEM-MDM/{input_var}_{year}.nc'
        feature = xr.open_dataset(input_file)
        if input_var.startswith('monthly'):
            feature = feature[input_vars_names[i]].values
            feature = np.nan_to_num(feature, nan=0.0)
            feature = np.repeat(feature[:, :, np.newaxis], 31, axis=2)
        else:
            feature = feature[input_vars_names[i]].values
        feature = feature.reshape(-1, 12, 31)
        feature = convert2year(feature)
        X_data.append(feature)
    X_data = np.stack(X_data, axis=-1)
    
    output_file = f'../../data/TEM-MDM/CH4_emission_intensity_{year}.nc'
    label = xr.open_dataset(output_file, engine="netcdf4")
    y_data = convert2year(label['CH4_emission'].values.reshape(-1,12,31))

    X_data = X_data[~mask]
    y_data = y_data[~mask]

    shape_masked = X_data.shape
    # print(shape_masked)

    # print(mask.shape)
    # print(np.sum(~mask))
    # print(X_data[0][0][0])
    # print(X_data[0][364][0])
    # print(X_data[1][0][0])
    # print(X_data[2][0][0])
    # print(y_data.shape)

    index = X_data.shape[0]
    selected = np.random.choice(np.arange(index), int(index*0.1), replace=False)
    temp_input = X_data[selected]
    temp_output = y_data[selected]
    temp_mask = ~(np.isnan(temp_input).any(axis=(1,2)) | np.isnan(temp_output).any(axis=1))
    temp_input = temp_input[temp_mask]
    temp_output = temp_output[temp_mask]
    # print(temp_input[0][0][0])
    print(temp_input.shape, temp_output.shape)
    np.save(f'../../processed_data/TEM-MDM/temporal/input_{year}.npy', temp_input)
    np.save(f'../../processed_data/TEM-MDM/temporal/output_{year}.npy', temp_output)

    for i, fold in enumerate(fold_index):
        fold_input, fold_output = X_data[fold], y_data[fold]
        index = fold_input.shape[0]
        selected = np.random.choice(np.arange(index), int(index*0.1), replace=False)
        fold_input = fold_input[selected]
        fold_output = fold_output[selected]
        fold_mask = ~(np.isnan(fold_input).any(axis=(1,2)) | np.isnan(fold_output).any(axis=1))
        fold_input = fold_input[fold_mask]
        fold_output = fold_output[fold_mask]
        print(fold_input.shape, fold_output.shape)
        np.save(f'../../processed_data/TEM-MDM/spatial/input_{year}_{i}.npy', fold_input)
        np.save(f'../../processed_data/TEM-MDM/spatial/output_{year}_{i}.npy', fold_output)

1979
(5815, 365, 15) (5815, 365)
(1149, 365, 15) (1149, 365)
(1154, 365, 15) (1154, 365)
(1157, 365, 15) (1157, 365)
(1164, 365, 15) (1164, 365)
(1167, 365, 15) (1167, 365)
1980
(5790, 365, 15) (5790, 365)
(1174, 365, 15) (1174, 365)
(1168, 365, 15) (1168, 365)
(1156, 365, 15) (1156, 365)
(1153, 365, 15) (1153, 365)
(1166, 365, 15) (1166, 365)
1981
(5812, 365, 15) (5812, 365)
(1155, 365, 15) (1155, 365)
(1160, 365, 15) (1160, 365)
(1152, 365, 15) (1152, 365)
(1159, 365, 15) (1159, 365)
(1159, 365, 15) (1159, 365)
1982
(5820, 365, 15) (5820, 365)
(1154, 365, 15) (1154, 365)
(1161, 365, 15) (1161, 365)
(1160, 365, 15) (1160, 365)
(1162, 365, 15) (1162, 365)
(1162, 365, 15) (1162, 365)
1983
(5817, 365, 15) (5817, 365)
(1155, 365, 15) (1155, 365)
(1154, 365, 15) (1154, 365)
(1159, 365, 15) (1159, 365)
(1168, 365, 15) (1168, 365)
(1166, 365, 15) (1166, 365)
1984
(5819, 365, 15) (5819, 365)
(1164, 365, 15) (1164, 365)
(1163, 365, 15) (1163, 365)
(1161, 365, 15) (1161, 365)
(1160, 365, 15) (1

In [14]:
index = shape_masked[0]
selected = np.random.choice(np.arange(index), int(index*0.1), replace=False)

for year in range(1979, 2019):
    print(year)
    X_data = []
    for feat in stat_dict.keys():
        feature = stat_dict[feat]
        feature = np.repeat(feature[:, np.newaxis], 365, axis=1)
        X_data.append(feature)
    feature = np.full(shape, ch4_dict[year])
    X_data.append(feature)
    feature = np.full(shape, co2_dict[year])
    X_data.append(feature)
    for i,input_var in enumerate(input_vars):
        input_file = f'../../data/TEM-MDM/{input_var}_{year}.nc'
        feature = xr.open_dataset(input_file)
        if input_var.startswith('monthly'):
            feature = feature[input_vars_names[i]].values
            feature = np.nan_to_num(feature, nan=0.0)
            feature = np.repeat(feature[:, :, np.newaxis], 31, axis=2)
        else:
            feature = feature[input_vars_names[i]].values
        feature = feature.reshape(-1, 12, 31)
        feature = convert2year(feature)
        X_data.append(feature)
    X_data = np.stack(X_data, axis=-1)
    
    output_file = f'../../data/TEM-MDM/CH4_emission_intensity_{year}.nc'
    label = xr.open_dataset(output_file, engine="netcdf4")
    y_data = convert2year(label['CH4_emission'].values.reshape(-1,12,31))

    X_data = X_data[~mask]
    y_data = y_data[~mask]

    temp_input = X_data[selected]
    temp_output = y_data[selected]
    temp_mask = ~(np.isnan(temp_input).any(axis=(1,2)) | np.isnan(temp_output).any(axis=1))
    temp_input = temp_input[temp_mask]
    temp_output = temp_output[temp_mask]
    print("year", year, "cle data:", temp_input[0][0][0])
    print(temp_input.shape, temp_output.shape)
    np.save(f'../../processed_data/TEM-MDM/sequence/temporal/input_{year}.npy', temp_input)
    np.save(f'../../processed_data/TEM-MDM/sequence/temporal/output_{year}.npy', temp_output)

    # for i, fold in enumerate(fold_index):
    #     fold_input, fold_output = X_data[fold], y_data[fold]
    #     index = fold_input.shape[0]
    #     selected = np.random.choice(np.arange(index), int(index*0.1), replace=False)
    #     fold_input = fold_input[selected]
    #     fold_output = fold_output[selected]
    #     fold_mask = ~(np.isnan(fold_input).any(axis=(1,2)) | np.isnan(fold_output).any(axis=1))
    #     fold_input = fold_input[fold_mask]
    #     fold_output = fold_output[fold_mask]
    #     print(fold_input.shape, fold_output.shape)
    #     np.save(f'../../processed_data/TEM-MDM/sequence/spatial/input_{year}_{i}.npy', fold_input)
    #     np.save(f'../../processed_data/TEM-MDM/sequence/spatial/output_{year}_{i}.npy', fold_output)

1979
year 1979 cle data: 173.0
(5819, 365, 15) (5819, 365)
1980
year 1980 cle data: 173.0
(5819, 365, 15) (5819, 365)
1981
year 1981 cle data: 173.0
(5819, 365, 15) (5819, 365)
1982
year 1982 cle data: 173.0
(5819, 365, 15) (5819, 365)
1983
year 1983 cle data: 173.0
(5819, 365, 15) (5819, 365)
1984
year 1984 cle data: 173.0
(5819, 365, 15) (5819, 365)
1985
year 1985 cle data: 173.0
(5819, 365, 15) (5819, 365)
1986
year 1986 cle data: 173.0
(5819, 365, 15) (5819, 365)
1987
year 1987 cle data: 173.0
(5819, 365, 15) (5819, 365)
1988
year 1988 cle data: 173.0
(5819, 365, 15) (5819, 365)
1989
year 1989 cle data: 173.0
(5819, 365, 15) (5819, 365)
1990
year 1990 cle data: 173.0
(5819, 365, 15) (5819, 365)
1991
year 1991 cle data: 173.0
(5819, 365, 15) (5819, 365)
1992
year 1992 cle data: 173.0
(5819, 365, 15) (5819, 365)
1993
year 1993 cle data: 173.0
(5819, 365, 15) (5819, 365)
1994
year 1994 cle data: 173.0
(5819, 365, 15) (5819, 365)
1995
year 1995 cle data: 173.0
(5819, 365, 15) (5819, 36